# GloVe: Global Vectors for Word Representation

Replication of Pennington, Socher and Manning (2014), *GloVe: Global Vectors for Word
Representation*, EMNLP.

Unlike prediction-based methods, GloVe learns word vectors by directly factorizing the
global word-word co-occurrence matrix: it fits the dot product of two word vectors to the
log of how often the words co-occur, weighted so that rare and very frequent pairs do not
dominate. We build a co-occurrence matrix from a small public-domain corpus, minimize the
GloVe weighted least-squares objective, and reproduce the result that the learned vectors
place semantically related words near one another.

In [1]:
import re, urllib.request, collections, numpy as np, torch
torch.manual_seed(0)

In [2]:
# Small public-domain corpus (Project Gutenberg).
urls = ["https://www.gutenberg.org/files/11/11-0.txt",
        "https://www.gutenberg.org/files/12/12-0.txt",
        "https://www.gutenberg.org/files/16/16-0.txt"]
text = " ".join(urllib.request.urlopen(u, timeout=30).read().decode("utf-8", "ignore") for u in urls)
words = re.findall(r"[a-z]+", text.lower())
counts = collections.Counter(words)
vocab = [w for w, c in counts.most_common(3000)]
stoi = {w: i for i, w in enumerate(vocab)}; itos = {i: w for w, i in stoi.items()}
data = [stoi[w] for w in words if w in stoi]
Vn = len(vocab); print("tokens", len(data), "vocab", Vn)

tokens 102231 vocab 3000


In [3]:
# Build the co-occurrence matrix with 1/distance weighting inside a symmetric window.
WINDOW = 5
cooc = collections.defaultdict(float)
for i, wi in enumerate(data):
    for d in range(1, WINDOW + 1):
        if i - d >= 0:
            wj = data[i-d]; cooc[(wi, wj)] += 1.0/d; cooc[(wj, wi)] += 1.0/d
rows, cols, vals = [], [], []
for (i, j), v in cooc.items():
    rows.append(i); cols.append(j); vals.append(v)
rows = torch.tensor(rows); cols = torch.tensor(cols); vals = torch.tensor(vals)
print("non-zero co-occurrence entries:", len(vals))

non-zero co-occurrence entries: 306413


In [4]:
# GloVe objective: f(X) * (w_i . w~_j + b_i + b~_j - log X)^2, with f the weighting function.
DIM, XMAX, A = 80, 100.0, 0.75
W  = torch.nn.Parameter(torch.randn(Vn, DIM) * 0.1)
Wc = torch.nn.Parameter(torch.randn(Vn, DIM) * 0.1)
b  = torch.nn.Parameter(torch.zeros(Vn)); bc = torch.nn.Parameter(torch.zeros(Vn))
opt = torch.optim.Adam([W, Wc, b, bc], lr=0.05)
fw = torch.clamp((vals / XMAX) ** A, max=1.0)       # weighting f(X_ij)
logx = torch.log(vals)
for ep in range(60):
    pred = (W[rows] * Wc[cols]).sum(1) + b[rows] + bc[cols]
    loss = (fw * (pred - logx) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep+1) % 15 == 0: print(f"epoch {ep+1}: weighted MSE = {loss.item():.4f}")

epoch 15: weighted MSE = 0.0130


epoch 30: weighted MSE = 0.0054


epoch 45: weighted MSE = 0.0032


epoch 60: weighted MSE = 0.0023


In [5]:
# Nearest neighbors using the combined vectors W + Wc (as recommended in the paper).
emb = (W + Wc).detach()
emb = emb / emb.norm(dim=1, keepdim=True)
def neighbors(word, k=8):
    if word not in stoi: return f"'{word}' not in vocab"
    sims = emb @ emb[stoi[word]]
    return [itos[i.item()] for i in sims.argsort(descending=True)[1:k+1]]
for w in ["king", "queen", "little", "rabbit", "garden"]:
    print(f"{w:8s} ->", neighbors(w))

king     -> ['queen', 'red', 'knight', 'hatter', 'white', 'mock', 'turtle', 'gryphon']
queen    -> ['red', 'white', 'king', 'knight', 'said', 'alice', 'the', 'rabbit']
little   -> ['a', 'house', 'in', 'of', 'girl', 'very', 'and', 'the']
rabbit   -> ['white', 'drawer', 'king', 'red', 'hole', 'dodo', 'kitten', 'black']
garden   -> ['duchess', 'lion', 'conclusion', 'fireplace', 'redskin', 'lizard', 'crocodile', 'spot']
